## Reading and Preparing a Research PDF for RAG using LangChain

In [ ]:
#%run /Users/csharm33/code/genai_dbx/Course3/.setup/learner_setup.ipynb


## Common Word Embedding Techniques

There are several ways to generate embeddings. Let’s look at two widely used methods:

### a) Word2Vec
- A traditional method using shallow neural networks.
- Captures the relationship between words based on **how often they appear together**.
- Learns embeddings like: “king - man + woman = queen”.



### b) AzureOpenAI using Langchain Embeddings

OpenAI provides powerful pre-trained models for creating embeddings using their API.
    

### Most Common Model:
- `text-embedding-ada-002` — small, fast, and very good.
-  It produces a **1536-dimensional vector** for each chunk.
-  Ideal for search, retrieval, and similarity tasks.

### How It Works:
- You send a chunk of text to AzureOpenAI Endpoint.
- The model returns a fixed-length embedding (vector) that captures the **semantic meaning** of the text.

AzureOpenAI using Langchain embeddings are **widely used in production RAG pipelines**, especially when quality and ease-of-use matter.



In [ ]:
import numpy as np
import os
import httpx
import openai
from dotenv import load_dotenv
import nltk
from nltk.tokenize import sent_tokenize
from gensim.models import Word2Vec
from langchain_openai import AzureOpenAIEmbeddings
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
)
import warnings
warnings.filterwarnings("ignore")

In [ ]:
nltk.data.path.append("/Users/csharm33/code/genai_dbx/Course3/Module2/Data/nltk/nltk_data")

In [ ]:
# Authentication:
def get_access_token():
    auth = "https://api.uhg.com/oauth2/token"
    scope = "https://api.uhg.com/.default"
    grant_type = "client_credentials"


    with httpx.Client() as client:
        body = {
            "grant_type": grant_type,
            "scope": scope,
            "client_id": client_id,
            "client_secret": client_secret,
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        resp = client.post(auth, headers=headers, data=body, timeout=60)
        access_token = resp.json()["access_token"]
        return access_token


load_dotenv('/Users/csharm33/code/genai_dbx/Course3/Module2/Data/vars.env')

endpoint = os.environ.get("MODEL_ENDPOINT")
EMBEDDINGS_DEPLOYMENT_NAME = os.environ.get("EMBEDDINGS_MODEL_NAME")
project_id = os.environ.get("PROJECT_ID")
api_version = os.environ.get("API_VERSION")
client_id = os.environ.get("CLIENT_ID")
client_secret = os.environ.get("CLIENT_SECRET")


embeddings_client = openai.AzureOpenAI(
        azure_endpoint=endpoint,
        api_version=api_version,
        azure_deployment=EMBEDDINGS_DEPLOYMENT_NAME,
        azure_ad_token=get_access_token(),
        default_headers={
            "projectId": project_id
        }
    )


In [ ]:
# this function returns vector embeddings for the provided text chunks.

@retry(wait=wait_random_exponential(min=45, max=120), stop=stop_after_attempt(6))
def get_embeddings(texts_chunk):
    return embeddings_client.embeddings.create(input=texts_chunk, model=EMBEDDINGS_DEPLOYMENT_NAME).data

In [ ]:
# Define a function to load and extract text from PDF
def load_pdf_with_langchain(pdf_path):
    
    # Use LangChain's built-in loader
    loader = PyMuPDFLoader(pdf_path)

    # Load the PDF into LangChain's document format
    documents = loader.load()

    print(f"Successfully loaded {len(documents)} document chunks from the PDF.")
    return documents

In [ ]:
# Path to the uploaded PDF (replace with your actual file path)
pdf_path = "/Users/csharm33/code/genai_dbx/Course3/Module2/Data/HealthcaredocforRAG.pdf"  

# Extract the document chunks
docs = load_pdf_with_langchain(pdf_path)


#### Let's Proceed with Sentence-Based Chunking

We will now apply the **sentence-based chunking approach** we discussed earlier, which is particularly useful for preparing text inputs for **Word2Vec** embeddings.


In [ ]:
def sentence_based_chunking(docs, sentences_per_chunk=3):
    chunks = []
    for doc in docs:
        sentences = sent_tokenize(doc.page_content)
        for i in range(0, len(sentences), sentences_per_chunk):
            chunk = " ".join(sentences[i:i + sentences_per_chunk])
            chunks.append(chunk)
    return chunks

# Generate sentence chunks
text_chunks = sentence_based_chunking(docs)


## Word Embeddings Using Word2Vec

We’ll use Gensim’s Word2Vec model to generate embeddings.

Note: Word2Vec works best on **large corpora**. For demo purposes, we’ll train it on our PDF chunks, but in practice you should use a pre-trained model.


In [ ]:
# Define a function to train Word2Vec on the given chunks and returns vector averages for each chunk.
def word2vec_embedding(chunks):
    
    # Tokenize each chunk into words
    tokenized = [chunk.split() for chunk in chunks]

    # Train a Word2Vec model
    model = Word2Vec(sentences=tokenized, vector_size=100, window=5, min_count=1, workers=2)

    embeddings = []
    for words in tokenized:
        vectors = [model.wv[word] for word in words if word in model.wv]
        # Take average vector for each chunk
        chunk_vector = np.mean(vectors, axis=0) if vectors else np.zeros(100)
        embeddings.append(chunk_vector)

    return embeddings
    
# Run Word2Vec embeddings
w2v_embeddings = word2vec_embedding(text_chunks)
print(f" Generated {len(w2v_embeddings)} Word2Vec chunk embeddings")
print(f" Generated Word2Vec first chunk embeddings dimension {w2v_embeddings[0].shape} ")

In [ ]:
w2v_embeddings[0][0:10]


### b) AzureOpenAI using Langchain Embeddings

OpenAI provides powerful pre-trained models for creating embeddings using their API

In [ ]:
# Step 1: Load and chunk using RecursiveCharacterTextSplitter
def get_recursive_chunks(pdf_path, chunk_size=500, chunk_overlap=50):
    loader = PyMuPDFLoader(pdf_path)
    raw_docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(raw_docs)

    # Extract just the text part for embedding
    return [chunk.page_content for chunk in chunks]


# Example: Running it all together
pdf_path = "/Users/csharm33/code/genai_dbx/Course3/Module2/Data/HealthcaredocforRAG.pdf"  
text_chunks = get_recursive_chunks(pdf_path)
openai_embeddings = get_embeddings(text_chunks)

print(f"Generated {len(openai_embeddings)} OpenAI embeddings.")
print(f"First chunk embedding size: {len(openai_embeddings[0].embedding)}")
print (f"First chunk embedding : {openai_embeddings[0].embedding[0:10]}")